In [ ]:
import pandas as pd

#train_df = pd.read_csv("/content/drive/MyDrive/financial_transactions_train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/financial_transactions_test.csv")

print(test_df.head())
print(test_df.shape)
print(test_df.columns)

                              Transaction_Text       Label
0  Mobile EMI payment - INR 45866 - Ref 955127         EMI
1  Education loan EMI - INR 32772 - Ref 858630         EMI
2       Bike loan EMI - INR 12852 - Ref 259771         EMI
3    PPF contribution - INR 10392 - Ref 505075  Investment
4     NPS contribution - INR 3583 - Ref 580994  Investment
(1000, 2)
Index(['Transaction_Text', 'Label'], dtype='object')


In [2]:
print("\nMissing values:")
print(test_df.isnull().sum())


Missing values:
Transaction_Text    0
Label               0
dtype: int64


In [3]:
print("\nDuplicate rows:")
print(test_df.duplicated().sum())


Duplicate rows:
0


In [4]:
print("\nClass distribution:")
print(test_df["Label"].value_counts())


Class distribution:
Label
Food          217
EMI           204
Shopping      204
Investment    189
Travel        186
Name: count, dtype: int64


In [5]:
import re

def clean_text(text):
    text = text.lower()

    # Remove INR amount
    text = re.sub(r'\binr\s*\d+(?:\.\d+)?\b', '', text)

    # Remove reference number
    text = re.sub(r'\bref\s*\d+\b', '', text)

    # Remove hyphens
    text = re.sub(r'-+', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

test_df["Clean_Text"] = test_df["Transaction_Text"].apply(clean_text)

print(test_df[["Transaction_Text", "Clean_Text", "Label"]].head(20))

                                  Transaction_Text             Clean_Text  \
0      Mobile EMI payment - INR 45866 - Ref 955127     mobile emi payment   
1      Education loan EMI - INR 32772 - Ref 858630     education loan emi   
2           Bike loan EMI - INR 12852 - Ref 259771          bike loan emi   
3        PPF contribution - INR 10392 - Ref 505075       ppf contribution   
4         NPS contribution - INR 3583 - Ref 580994       nps contribution   
5    Electronics purchase - INR 25775 - Ref 761604   electronics purchase   
6       Hotel dinner bill - INR 16135 - Ref 592344      hotel dinner bill   
7      IRCTC train ticket - INR 33490 - Ref 509896     irctc train ticket   
8    Zomato food delivery - INR 37155 - Ref 672579   zomato food delivery   
9    Myntra fashion order - INR 11689 - Ref 528365   myntra fashion order   
10    Metro card recharge - INR 21609 - Ref 859684    metro card recharge   
11          Appliance EMI - INR 24071 - Ref 609771          appliance emi   

In [6]:
test_df.head()

,Transaction_Text,Label,Clean_Text
0,Mobile EMI payment - INR 45866 - Ref 955127,EMI,mobile emi payment
1,Education loan EMI - INR 32772 - Ref 858630,EMI,education loan emi
2,Bike loan EMI - INR 12852 - Ref 259771,EMI,bike loan emi
3,PPF contribution - INR 10392 - Ref 505075,Investment,ppf contribution
4,NPS contribution - INR 3583 - Ref 580994,Investment,nps contribution


In [7]:
from sklearn.model_selection import train_test_split

X = test_df["Clean_Text"]
y = test_df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 800
Testing samples: 200


**TF-IDF (Term Frequency-Inverse Document Frequency)** is a statistical score that evaluates how important a word is to a document in a collection (corpus) of documents.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2)
)

In [10]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [11]:
print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (800, 179)
Testing shape: (200, 179)


In [12]:
print("\nVocabulary:")
print(vectorizer.get_feature_names_out())


Vocabulary:
['account' 'account transfer' 'airport' 'airport taxi' 'ajio'
 'ajio clothing' 'amazon' 'amazon online' 'appliance' 'appliance emi'
 'bakery' 'bakery purchase' 'bike' 'bike loan' 'bike ride' 'bill' 'bond'
 'bond investment' 'booking' 'bus' 'bus ticket' 'cab' 'cab booking' 'cafe'
 'cafe coffee' 'car' 'car loan' 'card' 'card emi' 'card recharge'
 'clothing' 'clothing purchase' 'coffee' 'coffee purchase' 'consumer'
 'consumer durable' 'contribution' 'credit' 'credit card' 'debit'
 'delivery' 'demat' 'demat account' 'deposit' 'deposit investment' 'dine'
 'dine in' 'dinner' 'dinner bill' 'dominos' 'dominos pizza' 'durable'
 'durable emi' 'ebay' 'ebay online' 'education' 'education loan'
 'electronics' 'electronics purchase' 'emi' 'emi payment' 'etf'
 'etf purchase' 'fare' 'fashion' 'fashion order' 'fixed' 'fixed deposit'
 'flight' 'flight ticket' 'flipkart' 'flipkart purchase' 'food'
 'food delivery' 'food items' 'fund' 'fund investment' 'fund sip' 'gold'
 'gold bond' 'grocery'

In [13]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [14]:
y_pred = lr_model.predict(X_test_tfidf)
print(y_pred[:20])

['EMI' 'Travel' 'Food' 'Investment' 'Travel' 'Shopping' 'Shopping' 'EMI'
 'Food' 'EMI' 'Food' 'Shopping' 'EMI' 'Travel' 'Food' 'Food' 'EMI'
 'Investment' 'Travel' 'Food']


In [15]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

         EMI       1.00      1.00      1.00        41
        Food       1.00      1.00      1.00        43
  Investment       1.00      1.00      1.00        38
    Shopping       1.00      1.00      1.00        41
      Travel       1.00      1.00      1.00        37

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


Confusion Matrix:
[[41  0  0  0  0]
 [ 0 43  0  0  0]
 [ 0  0 38  0  0]
 [ 0  0  0 41  0]
 [ 0  0  0  0 37]]


In [17]:
new_transactions = [
    "mobile phone emi payment",
    "zomato dinner order",
    "myntra clothes purchase",
    "uber cab ride",
    "mutual fund investment"
]

new_clean = [clean_text(x) for x in new_transactions]

new_tfidf = vectorizer.transform(new_clean)

predictions = lr_model.predict(new_tfidf)

for text, prediction in zip(new_transactions, predictions):
    print(text, "→", prediction)

mobile phone emi payment → EMI
zomato dinner order → Food
myntra clothes purchase → Shopping
uber cab ride → Travel
mutual fund investment → Investment


In [18]:
print("Duplicate rows:", test_df.duplicated().sum())

print("Duplicate Transaction_Text:",
      test_df["Transaction_Text"].duplicated().sum())

print("Duplicate Clean_Text:",
      test_df["Clean_Text"].duplicated().sum())

Duplicate rows: 0
Duplicate Transaction_Text: 0
Duplicate Clean_Text: 950
